# SpatialData Prototype: Dual-Target Viewers for `SLIDE-0329_crop_2048`

This notebook keeps `cell_labels` as the canonical segmentation layer, keeps the Nimbus table linked to those labels, and derives `cell_boundaries` only as a viewer/export layer.

The notebook is organized around four checks:
1. build a canonical `SpatialData` object with `full_image`, `cell_labels`, optional `nuclear_labels`, and `nimbus_table`
2. verify exact ID agreement between mask labels, Nimbus rows, and raster aggregation
3. derive polygons with `spatialdata.to_polygons(...)`, run simple geometry QC, and attach them as a derived shapes layer
4. show how Xenium Explorer style IDs can be mapped back to the original Nimbus `cell_id`


In [1]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import anndata as ad
import numpy as np
import pandas as pd
import sopa
import spatialdata
import tifffile
from shapely import make_valid
from sopa.io.explorer.utils import str_cell_id
from spatialdata import SpatialData, aggregate, to_polygons
from spatialdata.models import Labels2DModel, ShapesModel, TableModel

CROP_OUTPUTS = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs")
FULL_MERGE_PATH = CROP_OUTPUTS / "SLIDE-0329_crop_2048_full_merge.ome.tif"
CELL_MASK_PATH = CROP_OUTPUTS / "masks" / "SLIDE-0329_crop_2048_whole_cell.tiff"
NUCLEAR_MASK_PATH = CROP_OUTPUTS / "masks" / "SLIDE-0329_crop_2048_nuclear.tiff"
NIMBUS_TABLE_PATH = CROP_OUTPUTS / "nimbus" / "cell_table_full.csv"
SCRATCH_DIR = CROP_OUTPUTS / "spatialdata_dual_target_preview"
WRITE_ODON_ZARR = True
WRITE_XENIUM_SHAPES = False
XENIUM_MAX_VERTICES = 32

required_paths = [FULL_MERGE_PATH, CELL_MASK_PATH, NIMBUS_TABLE_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f"Missing required inputs: {missing_paths}"

print({
    "spatialdata": spatialdata.__version__,
    "sopa": sopa.__version__,
    "outputs_root": str(CROP_OUTPUTS),
})


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'spatialdata': '0.7.2', 'sopa': '2.2.4', 'outputs_root': '/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs'}


## Setup and Inputs

Start from the existing crop artifacts only. No merge, segmentation, or Nimbus inference is rerun in this notebook.


In [2]:
image_only_sdata = sopa.io.ome_tif(FULL_MERGE_PATH)
full_image_key = next(iter(image_only_sdata.images.keys()))
full_image = image_only_sdata.images[full_image_key]

cell_mask = tifffile.imread(CELL_MASK_PATH)
cell_labels = Labels2DModel.parse(cell_mask, dims=("y", "x"))

has_nuclear_mask = NUCLEAR_MASK_PATH.exists()
if has_nuclear_mask:
    nuclear_mask = tifffile.imread(NUCLEAR_MASK_PATH)
    nuclear_labels = Labels2DModel.parse(nuclear_mask, dims=("y", "x"))
else:
    nuclear_mask = None
    nuclear_labels = None

nimbus_df = pd.read_csv(NIMBUS_TABLE_PATH)

scale0 = full_image["scale0"].image
print({
    "image_key": full_image_key,
    "image_shape": tuple(scale0.shape),
    "image_dims": tuple(scale0.dims),
    "cell_mask_shape": tuple(cell_mask.shape),
    "has_nuclear_mask": has_nuclear_mask,
    "nuclear_mask_shape": tuple(nuclear_mask.shape) if nuclear_mask is not None else None,
    "nimbus_shape": tuple(nimbus_df.shape),
})


{'image_key': 'SLIDE-0329_crop_2048_full_merge', 'image_shape': (24, 2048, 2048), 'image_dims': ('c', 'y', 'x'), 'cell_mask_shape': (2048, 2048), 'has_nuclear_mask': True, 'nuclear_mask_shape': (2048, 2048), 'nimbus_shape': (4459, 6)}


## Canonical SpatialData Object

Keep the canonical object labels-first for exact table linkage and exact raster aggregation.


In [3]:
assert cell_mask.shape == tuple(scale0.shape[-2:]), "Cell mask must match the scale0 image canvas"
if nuclear_mask is not None:
    assert nuclear_mask.shape == cell_mask.shape, "Nuclear mask must match the cell mask canvas"

assert "cell_id" in nimbus_df.columns, "Nimbus table must contain a cell_id column"
assert "fov" in nimbus_df.columns, "Nimbus table must contain an fov column"
assert nimbus_df["fov"].nunique() == 1, "Expected exactly one FOV in the crop Nimbus table"
assert nimbus_df["cell_id"].is_unique, "Nimbus cell_id values must be unique"

feature_columns = [column for column in nimbus_df.columns if column not in {"cell_id", "fov"}]
assert feature_columns, "Expected at least one numeric Nimbus feature column"
assert all(pd.api.types.is_numeric_dtype(nimbus_df[column]) for column in feature_columns), "All Nimbus feature columns must be numeric"

obs = nimbus_df[["cell_id", "fov"]].copy()
obs["instance_id"] = obs["cell_id"].astype(str)
obs["region"] = "cell_labels"
obs.index = obs["instance_id"]

nimbus_table = TableModel.parse(
    ad.AnnData(
        X=nimbus_df[feature_columns].to_numpy(dtype=float),
        obs=obs,
        var=pd.DataFrame(index=pd.Index(feature_columns, name="feature")),
    ),
    region="cell_labels",
    region_key="region",
    instance_key="instance_id",
)

base_labels = {"cell_labels": cell_labels}
if nuclear_labels is not None:
    base_labels["nuclear_labels"] = nuclear_labels

base_sdata = SpatialData(
    images={"full_image": full_image},
    labels=base_labels,
    tables={"nimbus_table": nimbus_table},
)

print({
    "images": list(base_sdata.images.keys()),
    "labels": list(base_sdata.labels.keys()),
    "tables": list(base_sdata.tables.keys()),
    "nimbus_fov": obs["fov"].iloc[0],
    "nimbus_feature_count": len(feature_columns),
})


{'images': ['full_image'], 'labels': ['cell_labels', 'nuclear_labels'], 'tables': ['nimbus_table'], 'nimbus_fov': 'SLIDE-0329_crop_2048', 'nimbus_feature_count': 4}


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/spatialdata/models/models.py:1206: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


In [4]:
mask_label_ids = np.unique(cell_mask)
mask_label_ids = mask_label_ids[mask_label_ids > 0]
mask_id_set = set(map(str, mask_label_ids))
nimbus_id_set = set(base_sdata.tables["nimbus_table"].obs["instance_id"].astype(str))

print({
    "mask_label_count": len(mask_id_set),
    "nimbus_row_count": len(nimbus_id_set),
    "exact_match": mask_id_set == nimbus_id_set,
    "mask_not_in_nimbus": len(mask_id_set - nimbus_id_set),
    "nimbus_not_in_mask": len(nimbus_id_set - mask_id_set),
})

assert mask_id_set == nimbus_id_set, "Mask labels and Nimbus rows must match exactly"


{'mask_label_count': 4459, 'nimbus_row_count': 4459, 'exact_match': True, 'mask_not_in_nimbus': 0, 'nimbus_not_in_mask': 0}


In [5]:
agg_labels_sdata = aggregate(
    values="full_image",
    by="cell_labels",
    values_sdata=base_sdata,
    by_sdata=base_sdata,
    agg_func="mean",
    table_name="agg_labels",
)
agg_labels_table = agg_labels_sdata.tables["agg_labels"]
agg_label_id_set = set(agg_labels_table.obs["instance_id"].astype(str))

print({
    "agg_labels_shape": tuple(agg_labels_table.shape),
    "agg_label_count": len(agg_label_id_set),
    "agg_matches_mask": agg_label_id_set == mask_id_set,
    "agg_matches_nimbus": agg_label_id_set == nimbus_id_set,
})

assert agg_label_id_set == mask_id_set == nimbus_id_set, "Raster aggregation must preserve the canonical IDs"


{'agg_labels_shape': (4459, 24), 'agg_label_count': 4459, 'agg_matches_mask': True, 'agg_matches_nimbus': True}


In [7]:
base_sdata

SpatialData object
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
└── Tables
      └── 'nimbus_table': AnnData (4459, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels)

In [8]:
base_sdata['full_image'].scale0.image

<xarray.DataArray 'image' (c: 24, y: 2048, x: 2048)> Size: 201MB
dask.array<rechunk-merge, shape=(24, 2048, 2048), dtype=uint16, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U20 2kB 'R1_CY3_AF_I' 'R1_CY5_AF_I' ... 'R4_GFP_POLY_AF488'
  * y        (y) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
  * x        (x) float64 16kB 0.5 1.5 2.5 3.5 ... 2.046e+03 2.046e+03 2.048e+03
Attributes:
    transform:  {'global': Identity }

In [12]:
odon_store

PosixPath('/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/spatialdata_dual_target_preview/odon_labels_first.sdata.zarr')

In [9]:
if WRITE_ODON_ZARR:
    odon_store = SCRATCH_DIR / "odon_labels_first.sdata.zarr"
    odon_store.parent.mkdir(parents=True, exist_ok=True)
    base_sdata.write(odon_store, overwrite=True)
    print(f"Wrote labels-first SpatialData store for Odon testing: {odon_store}")
else:
    print("WRITE_ODON_ZARR is False. Toggle it to write a labels-first SpatialData zarr for Odon.")


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/ome_zarr/writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


Wrote labels-first SpatialData store for Odon testing: /mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/spatialdata_dual_target_preview/odon_labels_first.sdata.zarr


## Derived Shapes Layer

Use `spatialdata.to_polygons(...)` to derive a shapes layer without changing the canonical label-linked object.


In [10]:
polygon_df = to_polygons(base_sdata.labels["cell_labels"]).copy()
polygon_df["cell_id"] = polygon_df["label"].astype(int)
polygon_df["instance_id"] = polygon_df["cell_id"].astype(str)
polygon_df = polygon_df.sort_values("cell_id").reset_index(drop=True)

invalid_before = int((~polygon_df.geometry.is_valid).sum())
if invalid_before:
    invalid_mask = ~polygon_df.geometry.is_valid
    polygon_df.loc[invalid_mask, "geometry"] = polygon_df.loc[invalid_mask, "geometry"].map(make_valid)
    invalid_mask = ~polygon_df.geometry.is_valid
    if invalid_mask.any():
        polygon_df.loc[invalid_mask, "geometry"] = polygon_df.loc[invalid_mask, "geometry"].buffer(0)

polygon_df = polygon_df[~polygon_df.geometry.is_empty].copy()
polygon_df["explorer_cell_id"] = [str_cell_id(i) for i in range(len(polygon_df))]
invalid_after = int((~polygon_df.geometry.is_valid).sum())

print({
    "polygon_count": len(polygon_df),
    "invalid_before": invalid_before,
    "invalid_after": invalid_after,
    "empty_after_cleanup": int(polygon_df.geometry.is_empty.sum()),
    "label_ids_match_mask": set(polygon_df["instance_id"]) == mask_id_set,
})

assert set(polygon_df["instance_id"]) == mask_id_set, "Vectorized polygons must preserve the original cell IDs"


{'polygon_count': 4459, 'invalid_before': 9, 'invalid_after': 0, 'empty_after_cleanup': 0, 'label_ids_match_mask': True}


In [11]:
cell_boundaries_df = polygon_df.set_index("instance_id", drop=False).copy()
cell_boundaries = ShapesModel.parse(cell_boundaries_df)

viewer_labels = {"cell_labels": cell_labels}
if nuclear_labels is not None:
    viewer_labels["nuclear_labels"] = nuclear_labels

viewer_sdata = SpatialData(
    images={"full_image": full_image},
    labels=viewer_labels,
    shapes={"cell_boundaries": cell_boundaries},
    tables={"nimbus_table": nimbus_table},
)

id_map_df = polygon_df[["cell_id", "instance_id", "explorer_cell_id"]].copy()
print(viewer_sdata)
print(id_map_df.head().to_string(index=False))


SpatialData object
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (4459, 5) (2D shapes)
└── Tables
      └── 'nimbus_table': AnnData (4459, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes)
 cell_id instance_id explorer_cell_id
      85          85       aaaaaaaa-1
      86          86       aaaaaaab-1
      87          87       aaaaaaac-1
      88          88       aaaaaaad-1
      89          89       aaaaaaae-1


## Xenium Explorer Style IDs

Sopa shapes aggregation is fast, but it rewrites row IDs into Explorer-style strings. The mapping table above lets us recover the original Nimbus-linked `cell_id` values.


In [ ]:
shape_agg_sdata = SpatialData(
    images={"full_image": full_image},
    shapes={"cell_boundaries": cell_boundaries},
)

sopa.aggregate(
    shape_agg_sdata,
    aggregate_genes=False,
    aggregate_channels=True,
    image_key="full_image",
    shapes_key="cell_boundaries",
    key_added="agg_shapes",
    min_intensity_ratio=0.0,
    drop_filtered_cells=False,
)

agg_shapes_table = shape_agg_sdata.tables["agg_shapes"]
explorer_map_df = id_map_df.set_index("explorer_cell_id")
recovered_instance_ids = explorer_map_df.loc[agg_shapes_table.obs_names, "instance_id"].astype(str)
shape_agg_id_set = set(recovered_instance_ids)

print({
    "agg_shapes_shape": tuple(agg_shapes_table.shape),
    "shape_agg_row_count": len(shape_agg_id_set),
    "recovered_matches_mask": shape_agg_id_set == mask_id_set,
    "first_shape_agg_ids": agg_shapes_table.obs_names[:5].tolist(),
    "first_recovered_ids": recovered_instance_ids.iloc[:5].tolist(),
})

assert shape_agg_id_set == mask_id_set, "Recovered Explorer IDs must map back to the canonical cell IDs"


In [ ]:
comparison_df = pd.DataFrame(
    {
        "mask_instance_id": sorted(mask_id_set, key=int),
        "nimbus_instance_id": sorted(nimbus_id_set, key=int),
        "agg_labels_instance_id": sorted(agg_label_id_set, key=int),
        "agg_shapes_instance_id": sorted(shape_agg_id_set, key=int),
    }
)
comparison_df["all_match"] = comparison_df.nunique(axis=1) == 1
print(comparison_df.head().to_string(index=False))
print({
    "all_rows_match": bool(comparison_df["all_match"].all()),
    "row_count": len(comparison_df),
})

assert comparison_df["all_match"].all(), "All canonical and recovered ID views should agree row-by-row"


In [ ]:
if WRITE_XENIUM_SHAPES:
    from sopa.io.explorer.shapes import write_polygons

    xenium_dir = SCRATCH_DIR / "xenium_preview"
    xenium_dir.mkdir(parents=True, exist_ok=True)
    explorer_shapes_df = polygon_df[["cell_id", "instance_id", "explorer_cell_id", "geometry"]].copy()
    explorer_shapes_df = explorer_shapes_df.set_index("explorer_cell_id")
    write_polygons(
        path=xenium_dir,
        geo_df=explorer_shapes_df,
        max_vertices=XENIUM_MAX_VERTICES,
        is_dir=True,
        preserve_ids=True,
    )
    id_map_df.to_csv(xenium_dir / "explorer_id_map.csv", index=False)
    print(f"Wrote Xenium preview artifacts to: {xenium_dir}")
else:
    print("WRITE_XENIUM_SHAPES is False. Toggle it to write simplified Xenium preview polygons and the ID map.")
